In [1]:
!pip install xarray zarr gcsfs netCDF4 dask numpy

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ---------------------- ----------------- 0.8/1.4 MB 139.4 MB/s eta 0:00:01
   ----------------------------- ---------- 1.0/1.4 MB 537.8 kB/s eta 0:00:01
   -------------------------

In [4]:
!pip install toolz

In [5]:
import os
import numpy as np
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

def main():
    # 1. РЕШЕНИЕ ПРОБЛЕМЫ ЗАВИСАНИЯ: 
    # Принудительно включаем однопоточный режим для сетевых запросов.
    # Это предотвращает deadlock в gcsfs при скачивании чанков.
    dask.config.set(scheduler='single-threaded')

    output_dir = 'data'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'era5_sample_2019.nc')

    gcs_path = 'gs://weatherbench2/datasets/era5/1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr'
    
    print("Подключение к датасету...")
    # Добавлен token='anon', чтобы не было спама про авторизацию
    ds = xr.open_zarr(gcs_path, consolidated=True, storage_options={'token': 'anon'})

    # 2. Фильтруем данные
    ds_2019 = ds.sel(time=slice('2019-01-01', '2019-12-31'))

    np.random.seed(42) 
    all_times = ds_2019.time.values
    # Для начала протестируйте на size=1 или size=5, прежде чем качать 128
    random_times = np.random.choice(all_times, size=128, replace=False)
    random_times.sort()
    
    # Вместо random.choice скачает данные с 1 января по 1 февраля (как раз 128 срезов каждые 6 часов)
    ds_sample = ds_2019.isel(time=slice(0, 128))

    # 3. Списки переменных
    surface_vars = [
        '2m_temperature', 'mean_sea_level_pressure', '10m_u_component_of_wind',
        '10m_v_component_of_wind', 'total_precipitation_6hr', 'sea_surface_temperature',
        'total_column_water_vapour', 'total_cloud_cover'
    ]

    atm_vars = [
        'temperature', 'u_component_of_wind', 'v_component_of_wind',
        'geopotential', 'specific_humidity'
    ]
    levels = [1000, 925, 850, 700]

    print("Формирование графа задач (подготовка переменных)...")
    ds_surface = ds_sample[surface_vars]
    ds_atm = ds_sample[atm_vars].sel(level=levels)
    ds_final = xr.merge([ds_surface, ds_atm])

    # 4. Скачивание и сохранение
    print(f"Начинается скачивание и запись в {output_file}...")
    
    # Убрали .load(), чтобы данные потоково писались на диск.
    # Оборачиваем в ProgressBar, чтобы видеть реальную полосу загрузки в процентах
    with ProgressBar():
        ds_final.to_netcdf(output_file, compute=True)
        
    print("\nЗагрузка успешно завершена!")

if __name__ == "__main__":
    main()

Подключение к датасету...
Формирование графа задач (подготовка переменных)...
Начинается скачивание и запись в data\era5_sample_2019.nc...
[########################################] | 100% Completed | 244.02 s

Загрузка успешно завершена!


In [1]:
import os
import numpy as np
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

def main():
    # Оставляем однопоточный режим для стабильности скачивания огромных чанков
    dask.config.set(scheduler='single-threaded')

    output_dir = 'data'
    os.makedirs(output_dir, exist_ok=True)
    
    # Назовем файл иначе, чтобы не перезаписать предыдущий
    output_file = os.path.join(output_dir, 'era5_highres_sample_2019.nc')

    # Путь к датасету высокого разрешения (0.25 градуса, 1440x721)
    gcs_path = 'gs://weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr'
    
    print("Подключение к датасету WeatherBench 2 высокого разрешения...")
    ds = xr.open_zarr(gcs_path, consolidated=True, storage_options={'token': 'anon'})

    # Фильтруем данные за 2019 год
    ds_2019 = ds.sel(time=slice('2019-01-01', '2019-12-31'))

    # Выбираем 128 случайных временных срезов
    np.random.seed(42) 
    all_times = ds_2019.time.values
    random_times = np.random.choice(all_times, size=128, replace=False)
    random_times.sort()
    
    ds_sample = ds_2019.sel(time=random_times)

    # Списки переменных
    surface_vars = [
        '2m_temperature',
        'mean_sea_level_pressure',
        '10m_u_component_of_wind',
        '10m_v_component_of_wind',
        'total_precipitation_6hr',
        'sea_surface_temperature',
        'total_column_water_vapour',
        'total_cloud_cover'
    ]

    atm_vars = [
        'temperature',
        'u_component_of_wind',
        'v_component_of_wind',
        'geopotential',
        'specific_humidity'
    ]
    
    levels = [1000, 925, 850, 700]

    print("Формирование графа задач (подготовка переменных 1440x721)...")
    ds_surface = ds_sample[surface_vars]
    ds_atm = ds_sample[atm_vars].sel(level=levels)
    ds_final = xr.merge([ds_surface, ds_atm])

    print(f"Ожидаемый размер скачиваемых данных: ~15 ГБ.")
    print(f"Начинается скачивание и запись в {output_file}...")
    print("ВНИМАНИЕ: Из-за случайной выборки дат этот процесс может занять от 1 до нескольких часов.")
    
    # Скачиваем и сохраняем с отображением прогресса
    with ProgressBar():
        ds_final.to_netcdf(output_file, compute=True)
        
    print("\nЗагрузка успешно завершена!")

if __name__ == "__main__":
    main()

Подключение к датасету WeatherBench 2 высокого разрешения...
Формирование графа задач (подготовка переменных 1440x721)...
Ожидаемый размер скачиваемых данных: ~15 ГБ.
Начинается скачивание и запись в data\era5_highres_sample_2019.nc...
ВНИМАНИЕ: Из-за случайной выборки дат этот процесс может занять от 1 до нескольких часов.
[########################################] | 100% Completed | 28m 17s

Загрузка успешно завершена!
